## llama.cpp GGUF backend — Kaggle T4 x2

Run cells in order.

In [ ]:
# CELL 1 — setup repo
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/N3iKos/llama-cpp-notebook"
REPO_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/llama-cpp-notebook")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))

from gguf_backend.shell import run

print("repo ready:", REPO_DIR)
print("module ready: gguf_backend")

In [ ]:
# CELL 2 — diagnostics
from gguf_backend.shell import run

run("nvidia-smi --query-gpu=index,name,memory.total,memory.free,driver_version,compute_cap --format=csv,noheader,nounits || true")
run("nvidia-smi topo -m || true")
run("nvcc --version || true")
run("df -h /kaggle/working /tmp || true")

In [ ]:
# CELL 3 — install prebuilt llama.cpp
from gguf_backend.installer import ensure_apt_tools, install_llama_cpp_prebuilt
from gguf_backend.shell import run

ROOT = "/kaggle/working"

ensure_apt_tools()
info = install_llama_cpp_prebuilt(ROOT, cuda_preference="12.8", force=False)

print(info)
run(f'"{info["server"]}" --version')
run(f'"{info["server"]}" --list-devices')

In [ ]:
# CELL 4 — download model and optional mmproj
import os
from gguf_backend.downloader import download_model_pair

MODEL_URL = "https://huggingface.co/ggml-org/Qwen2.5-VL-3B-Instruct-GGUF/resolve/main/Qwen2.5-VL-3B-Instruct-Q4_K_M.gguf"
MMPROJ_URL = "https://huggingface.co/ggml-org/Qwen2.5-VL-3B-Instruct-GGUF/resolve/main/mmproj-Qwen2.5-VL-3B-Instruct-Q8_0.gguf"

MODEL_DIR = "/kaggle/working/models/current"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

cfg = download_model_pair(MODEL_URL, MMPROJ_URL, MODEL_DIR, hf_token=HF_TOKEN, connections=16)
print(cfg)

In [ ]:
# CELL 5 — config, start server, warmup
import json
from pathlib import Path
from gguf_backend.server import ServerConfig, start_server

ROOT = "/kaggle/working"
cfg = json.loads(Path("/kaggle/working/model_config.json").read_text())

CTX_SIZE = 8192
SPLIT_MODE = "row"          # row, layer, none
FALLBACK_SPLIT_MODE = "layer"
TENSOR_SPLIT = "1,1"

BATCH_SIZE = 2048
UBATCH_SIZE = 512
PARALLEL = 1
FLASH_ATTN = True
CACHE_TYPE_K = "f16"
CACHE_TYPE_V = "f16"

IMAGE_MIN_TOKENS = None
IMAGE_MAX_TOKENS = None
CHAT_TEMPLATE_KWARGS = None  # example: '{"enable_thinking":true}'

server_cfg = ServerConfig(
    root_dir=ROOT,
    model_path=cfg["model_path"],
    mmproj_path=cfg.get("mmproj_path", ""),
    port=8080,
    alias="local-vl",
    ctx_size=CTX_SIZE,
    split_mode=SPLIT_MODE,
    fallback_split_mode=FALLBACK_SPLIT_MODE,
    tensor_split=TENSOR_SPLIT,
    batch_size=BATCH_SIZE,
    ubatch_size=UBATCH_SIZE,
    parallel=PARALLEL,
    flash_attn=FLASH_ATTN,
    cache_type_k=CACHE_TYPE_K,
    cache_type_v=CACHE_TYPE_V,
    image_min_tokens=IMAGE_MIN_TOKENS,
    image_max_tokens=IMAGE_MAX_TOKENS,
    chat_template_kwargs=CHAT_TEMPLATE_KWARGS,
    cuda_visible_devices="0,1",
)

server_info = start_server(server_cfg, warmup=True)
print(server_info)

In [ ]:
# CELL 6 — text response test
import json
from gguf_backend.client import chat

status, resp = chat(
    "http://127.0.0.1:8080",
    "local-vl",
    "Tulis satu kalimat bahwa backend siap digunakan.",
    max_tokens=80,
)

print("status:", status)
print(json.dumps(resp, indent=2, ensure_ascii=False)[:3000])

In [ ]:
# CELL 7 — tunnels
from gguf_backend.tunnel import start_tunnels

TUNNEL_MODE = "both"  # both, ngrok, cloudflare
NGROK_AUTHTOKEN = ""

if not NGROK_AUTHTOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        NGROK_AUTHTOKEN = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
    except Exception:
        NGROK_AUTHTOKEN = ""

urls = start_tunnels(
    8080,
    "/kaggle/working",
    mode=TUNNEL_MODE,
    ngrok_token=NGROK_AUTHTOKEN,
    fallback_cloudflare=True,
)

print(urls)

In [ ]:
# CELL 8 — stop/status utilities
from gguf_backend.server import stop_server
from gguf_backend.shell import run

# stop_server("/kaggle/working")
run("nvidia-smi --query-gpu=index,name,memory.used,memory.free,utilization.gpu,power.draw --format=csv,noheader,nounits || true")
run("cat /kaggle/working/llama_server.pid 2>/dev/null || true")